# Sample model outputs for clinical validation
This notebooks creates a random sample for clinical validation. We include 15 cases for each of the held-out scanners (30 in total) that were not part of the first round of clinical validation and where our semi-automatic method failed to produce useful masks.
In addition (manually done, not part of this notebook) all cases (12) where the clinician selected "Neither agree nor disagree" (worst observed label) for "The reconstructed colon accurately represents the patient’s colonic anatomy visible on the source CT images" are included.

In [24]:
# import libraries
import pandas as pd

In [39]:
samples_first_round = pd.read_csv("../../../data/processed/manual_annotation/samples_for_clinical_validation/sample_clinical_validation_sa_method.csv")

# extract filenames of sample used for clinical validation of SA-method
samples_first_round["File / Case ID"] = samples_first_round["File / Case ID"].str.replace(
    r"^(\d+)_pos-(.+)$",
    lambda m: f"colon_{int(m.group(1)):04d}-{m.group(2)}",
    regex=True
)
samples_first_round_filenames = samples_first_round["File / Case ID"]

# extract filenames from non-collapsed dataset
filenames_non_collapsed = pd.read_csv("../../../data/raw/metadata/filenames_non_collapsed_renamed.txt", names=["filenames"])
filenames_non_collapsed_list = list(filenames_non_collapsed["filenames"])
filenames_non_collapsed_list = [filename[:-4] if filename.endswith(".mha") else filename for filename in filenames_non_collapsed_list]

# extract all filenames of held-out data
held_out = []
with open("../../../data/processed/metadata/filenames_held_out_data.txt", "r") as file:
    ([held_out.append(line.rstrip().replace(".mha", "")) for line in file])
print(f"# of all held-out filenames: {len(held_out)}")
print(f"# of filenames in first round: {len(samples_first_round_filenames.to_list())}")


# remove samples_first_round_filenames all held-out filenames
filtered_held_out = []
for f in held_out:
    if f in samples_first_round_filenames.to_list() or f in filenames_non_collapsed_list:
        continue
    else:
        filtered_held_out.append(f)
    
print(f"# of remaining filenames of held-out set (should be -30): {len(filtered_held_out)}")

# of all held-out filenames: 109
# of filenames in first round: 60
# of remaining filenames of held-out set (should be -30): 56


In [40]:
# add scanner info to filenames
metadata = pd.read_csv("../../../data/raw/metadata/merged_ku_tcia.csv")

# check that one subject is not part of scanners from different manufacturer
metadata_manufacturer_count_by_subject = (metadata.groupby('new_sub_id')['Manufacturer']
         .apply(lambda x: list(x.unique()))
         .reset_index()
         .assign(count=lambda d: d['Manufacturer'].str.len())  ## added line
)
print(f"Following patients have a scan taken with scanners from several manufacturer: {len(metadata_manufacturer_count_by_subject[metadata_manufacturer_count_by_subject["count"]>1])}")

# create dataframe with remaining held-out data and add relevant metadata (subject id & Manufacturer)
df_held_out_remaining = pd.DataFrame({"filename": filtered_held_out})
df_held_out_remaining["Subject ID"] = df_held_out_remaining["filename"].str.extract(r"(\d{4})")
df_held_out_remaining["Scanner"] = df_held_out_remaining["Subject ID"].apply(lambda x: metadata.loc[metadata["new_sub_id"] == f"sub{x[1:]}", "Manufacturer"].values[0])
df_held_out_remaining["Scanner"].value_counts()

Following patients have a scan taken with scanners from several manufacturer: 0


Scanner
TOSHIBA    29
Philips    27
Name: count, dtype: int64

In [45]:
# randomly sample 15 rows per manufacturer
sampled_annotations = df_held_out_remaining.groupby("Scanner").apply(lambda x: x.sample(n=15, random_state=42)).reset_index(drop=True)
sampled_annotations["filename"].sort_values().to_csv("../../../data/processed/manual_annotation/samples_for_clinical_validation/sample_clinical_validation_model.csv", index=False, header=False)